# Assignment 4: Retrieval-Augmented Generation (RAG)

Build a RAG pipeline with LangChain over the PubMedQA dataset.

**Outline**
1. Dataset (PubMedQA)
2. LangChain LM
3. Document database (embeddings + chunking + vector store)
4. Full RAG pipeline (LCEL)
5. Evaluation (RAG vs. no-context baseline + retrieval inspection)

## 0. Install dependencies
Uncomment the next cell on a fresh environment.

In [1]:
%pip install -q langchain langchain-community langchain-huggingface langchain-core sentence_transformers langchain-chroma transformers accelerate scikit-learn pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Part 1: The dataset
### Task 1.1. Download and inspect PubMedQA

In [2]:
import os, urllib.request

URL = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
if not os.path.exists("ori_pqal.json"):
    urllib.request.urlretrieve(URL, "ori_pqal.json")
print("file size (MB):", round(os.path.getsize("ori_pqal.json") / 1e6, 2))

file size (MB): 2.58


In [3]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T
# keep only yes/no labels (drop "maybe")
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    "year": tmp_data.YEAR,
})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index,
})

print("#documents:", len(documents), "  #questions:", len(questions))
print("label balance:\n", questions.gold_label.value_counts())

#documents: 890   #questions: 890
label balance:
 gold_label
yes    552
no     338
Name: count, dtype: int64


In [4]:
# Sanity check
print("Q:", questions.iloc[0].question)
print("\nGold label:", questions.iloc[0].gold_label)
print("\nDoc 0 abstract (first 500 chars):\n", documents.iloc[0].abstract[:500])

Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Gold label: yes

Doc 0 abstract (first 500 chars):
 Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


## Part 2: Configure your LangChain LM
### Task 2.1. Select a language model

We use a small open instruction-tuned model so the notebook runs locally without gated access. Swap `MODEL_ID` for e.g. `meta-llama/Llama-3.2-1B-Instruct` if you have a HF token (set env var `HUGGINGFACEHUB_API_TOKEN`).

In [5]:
import os, pathlib
os.environ["HF_TOKEN"] = pathlib.Path("~/.hf_token").expanduser().read_text().strip()

In [6]:
import os
print("HF_TOKEN set?", bool(os.environ.get("HF_TOKEN")))
print("first/last chars:", os.environ.get("HF_TOKEN","")[:5], "...", os.environ.get("HF_TOKEN","")[-4:])

HF_TOKEN set? True
first/last chars: hf_VE ... MEUU


In [ ]:
# this code is not run, according to the reason after this cell.
import torch
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # small + open. Try Llama-3.2-1B-Instruct, TinyLlama, etc.

device = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else -1)

llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    device=device,
    pipeline_kwargs={
        "max_new_tokens": 64,
        "do_sample": False,
        "return_full_text": False,
    },
)

# Sanity check
print(llm.invoke("Q: What is the capital of Sweden?\nA:"))

## Why an alternative approach is taken:
Because HuggingFace Hub throttled the model download to a stall (~11 MB in 21 min) despite valid authentication, so I switched to Ollama, which runs gpt-oss:20b locally from a pre-downloaded, 4-bit quantized weights via llama.cpp's Metal-accelerated resident server — no network, no HF, and much faster generation on Apple Silicon.

Try later with different network

#### Alternative: use a local Ollama model instead of `HuggingFacePipeline`


In [8]:
%pip install -q langchain-ollama


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_ollama import ChatOllama

MODEL_ID = "llama3:latest"  # any tag from `ollama list`

# Rebinds `llm` so the RAG chain (Part 4) and baseline chain (Part 5) use Ollama.
llm = ChatOllama(
    model=MODEL_ID,
    temperature=0.0,        # deterministic for evaluation
    num_predict=64,         # max new tokens (small — we only need yes/no)
    # base_url="http://localhost:11434",  # default; override if Ollama runs elsewhere
)

# Sanity check — ChatOllama returns an AIMessage; use .content for the string.
resp = llm.invoke("Q: What is the capital of Sweden?\nA:")
print(resp.content)

Stockholm!


## Part 3: Set up the document database
### Task 3.1. Embedding model

In [10]:
# Ollama embedding model (drop-in for HuggingFaceEmbeddings; same embed_query / embed_documents interface).
# Prereq: `ollama pull nomic-embed-text`
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    # base_url="http://localhost:11434",  # default; override if Ollama runs elsewhere
)

vec = embeddings.embed_query("What is programmed cell death?")
print("embedding shape:", (len(vec),))

embedding shape: (768,)


## Reflection:
means one embedding vector with 768 numbers that summarizes the whole input text.

In [11]:
# Cosine-similarity sanity check on 4 sentences
import numpy as np

sentences = [
    "The cat sat on the mat.",
    "A kitten napped on the rug.",
    "Programmed cell death regulates tumor growth.",
    "Apoptosis controls cancer progression.",
]

# embed_documents returns list[list[float]] with shape (n_sentences, 768)
X = np.array(embeddings.embed_documents(sentences))
print("matrix shape:", X.shape)

# nomic-embed-text vectors are already unit-normalized, but we renormalize defensively
X = X / np.linalg.norm(X, axis=1, keepdims=True)

# Cosine similarity for unit vectors is just the dot product
sim = X @ X.T

print("\nCosine-similarity matrix (rows/cols = sentence index):")
print(np.round(sim, 3))

print("\nPairwise similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"  s{i} vs s{j}: {sim[i, j]:.3f}   "
              f"| {sentences[i][:35]!r:38s} <-> {sentences[j][:35]!r}")

matrix shape: (4, 768)

Cosine-similarity matrix (rows/cols = sentence index):
[[1.    0.645 0.276 0.317]
 [0.645 1.    0.327 0.334]
 [0.276 0.327 1.    0.772]
 [0.317 0.334 0.772 1.   ]]

Pairwise similarities:
  s0 vs s1: 0.645   | 'The cat sat on the mat.'              <-> 'A kitten napped on the rug.'
  s0 vs s2: 0.276   | 'The cat sat on the mat.'              <-> 'Programmed cell death regulates tum'
  s0 vs s3: 0.317   | 'The cat sat on the mat.'              <-> 'Apoptosis controls cancer progressi'
  s1 vs s2: 0.327   | 'A kitten napped on the rug.'          <-> 'Programmed cell death regulates tum'
  s1 vs s3: 0.334   | 'A kitten napped on the rug.'          <-> 'Apoptosis controls cancer progressi'
  s2 vs s3: 0.772   | 'Programmed cell death regulates tum'  <-> 'Apoptosis controls cancer progressi'


### Task 3.2. Chunking

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

metadatas = [{"id": idx} for idx in documents.index]
raw_docs = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
chunks = text_splitter.split_documents(raw_docs)

print("#chunks:", len(chunks))
for c in chunks[:2]:
    print("---", c.metadata)
    print(c.page_content[:], "\n")

#chunks: 2406
--- {'id': 21645374}
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis 

--- {'id': 21645374}
. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window s

In [13]:
tail = chunks[0].page_content[-120:]
head = chunks[1].page_content[:120]
print("TAIL of chunk 0:\n", repr(tail))
print("HEAD of chunk 1:\n", repr(head))

TAIL of chunk 0:
 ' paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis'
HEAD of chunk 1:
 '. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on '


## My Reflection:
In this case only "madagascariensis" is common which is very short as overlap. Hence, the overlap size needs to be adjusted. I increased it from 100 to 200. 

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

metadatas = [{"id": idx} for idx in documents.index]
raw_docs = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
chunks = text_splitter.split_documents(raw_docs)

print("#chunks:", len(chunks))
for c in chunks[:2]:
    print("---", c.metadata)
    print(c.page_content[:], "\n")

#chunks: 2546
--- {'id': 21645374}
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis 

--- {'id': 21645374}
. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells th

In [15]:
tail = chunks[0].page_content[-220:]
head = chunks[1].page_content[:220]
print("TAIL of chunk 0:\n", repr(tail))
print("HEAD of chunk 1:\n", repr(head))

TAIL of chunk 0:
 'as been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis'
HEAD of chunk 1:
 '. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into thre'


## My Reflection
chunk size and overlap matter a lot for RAG. If chunks are too long, embeddings get less focused and can exceed the model context. If chunks are too short, one idea is split into pieces, so retrieval may return only part of the evidence. Overlap helps keep meaning when a sentence is cut between two chunks. Also, splitting with natural boundaries (like sentences/paragraphs) usually works better than fixed-size splitting.

From my examples, when overlap is too small, sometimes there is no meaningful shared text between neighbor chunks.

A conservative choice is overlap = longest sentence (often impractical). A more practical rule is overlap = 1.5 × average sentence length.

In [16]:
# This code provide Sentence-length statistics which can be used to better decide on the chunk_overlap parameter.
import numpy as np

sent_lengths = [
    len(s)
    for abstract in documents.abstract.tolist()
    for s in abstract.split(". ")
    if s.strip()
]

lengths = np.array(sent_lengths)

print(f"total sentences : {len(lengths):,}")
print(f"min length      : {lengths.min()} chars")
print(f"max length      : {lengths.max()} chars")
print(f"mean length     : {lengths.mean():.1f} chars")
print(f"median length   : {int(np.median(lengths))} chars")

print("\nSuggested chunk_overlap:")
print(f"  conservative (max):        {lengths.max()}")
print(f"  robust (95th percentile):  {int(np.percentile(lengths, 95))}")
print(f"  balanced (1.5x median):    {int(1.5 * np.median(lengths))}")

total sentences : 10,415
min length      : 1 chars
max length      : 730 chars
mean length     : 135.6 chars
median length   : 125 chars

Suggested chunk_overlap:
  conservative (max):        730
  robust (95th percentile):  260
  balanced (1.5x median):    187


## My Reflection:
chunk_size and separator parameters also have an effect on the quality of RAG. The design choice for them highly depends on the domain and data set.

For example, for chunk_size:

If in the document each sentence represents an independent piece of information, then retrieval of a single sentence might be more effective than longer sequences.
If the whole document presents a unique concept, it would be more effective to use the maximum possible chunk size (defined by the context window of the target LLM), as even a paragraph is hardly independent.

### Task 3.3. Vector store (Chroma, cosine similarity)

In [20]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pubmedqa",
    collection_metadata={"hnsw:space": "cosine"},
)

results = vector_store.similarity_search_with_score("What is programmed cell death?", k=3)
for res, score in results:
    print(f"* [SIM={score:.3f}] {res.page_content[:200]} ... [{res.metadata}]")

* [SIM=0.255] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co ... [{'id': 21645374}]
* [SIM=0.389] . A TUNEL assay showed fragmented nDNA in a gradient over these mitochondrial stages. Chloroplasts and transvacuolar strands were also examined using live cell imaging. The possible importance of mito ... [{'id': 21645374}]
* [SIM=0.391] . c-Kit was strongly expressed and tyrosine phosphorylated in cultured uveal melanoma cells compared with cutaneous melanoma cells. Moreover, in contrast to cutaneous melanoma cell lines c-kit maintai ... [{'id': 15223779}]


## Part 4: Implementing the system
### Task 4.1. Defining the full RAG pipeline (Option B — LCEL)

We use a `RunnableParallel` so the chain returns both the answer and the retrieved context for evaluation.

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever = vector_store.as_retriever(search_kwargs={"k": 1})

PROMPT_TEMPLATE = (
    "You are a medical question-answering assistant. "
    "Use ONLY the context below to answer the question with a single word: 'yes' or 'no'.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer (yes or no):"
)
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print("---- template ----")
print(prompt.messages[0].prompt.template)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

answer_chain = (
    {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

runnable_parallel = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
)
rag_chain = runnable_parallel.assign(answer=answer_chain)

# Sanity check
q = questions.iloc[0].question
out = rag_chain.invoke(q)
print("---- Sanity check ----")
print("Q:", q)
print("Gold:", questions.iloc[0].gold_label)
print("Answer:", out["answer"].strip())
print("Retrieved doc id:", out["context"][0].metadata)
print("Retrieved snippet:", out["context"][0].page_content[:300])

---- template ----
You are a medical question-answering assistant. Use ONLY the context below to answer the question with a single word: 'yes' or 'no'.

Context:
{context}

Question: {question}
Answer (yes or no):
---- Sanity check ----
Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold: yes
Answer: Yes
Retrieved doc id: {'id': 21645374}
Retrieved snippet: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel


## Part 5: Evaluate RAG on the dataset
### Task 5.1. High-level evaluation: RAG vs. no-context baseline

In [22]:
# Check what the LM actually returns on some random samples.
import random

random.seed(42)
n_samples = 20

# Pick 20 random questions
sample_indices = random.sample(range(len(questions)), n_samples)

# Store the raw outputs so we can count them at the end
outputs = []

for i in sample_indices:
    question_text = questions.iloc[i].question
    gold_answer = questions.iloc[i].gold_label

    result = rag_chain.invoke(question_text)
    raw_answer = result["answer"]

    outputs.append(raw_answer)

    print("index:", i, " | gold:", gold_answer, " | length:", len(raw_answer))
    print("raw:", repr(raw_answer))
    print("-" * 80)

# Count how many are already a clean yes/no
clean = 0
for text in outputs:
    cleaned = text.strip().lower().rstrip(".!?")
    if cleaned == "yes" or cleaned == "no":
        clean += 1

print("Clean yes/no answers:", clean, "out of", len(outputs))
print("Answers needing extra parsing:", len(outputs) - clean)

index: 654  | gold: no  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 114  | gold: yes  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 25  | gold: yes  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 759  | gold: yes  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 281  | gold: no  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 250  | gold: no  | length: 2
raw: 'No'
--------------------------------------------------------------------------------
index: 228  | gold: yes  | length: 3
raw: 'Yes'
--------------------------------------------------------------------------------
index: 142  | gold: yes  | length: 3
raw: 'Yes'
------------------------------------------------------

In [23]:
import re

def parse_yes_no(text: str):
    """Return 'yes', 'no', or None if no clear answer."""
    if text is None:
        return None
    m = re.search(r"\b(yes|no)\b", text.strip().lower())
    return m.group(1) if m else None

In [24]:
# No-context baseline chain
BASELINE_TEMPLATE = (
    "You are a medical question-answering assistant. "
    "Answer the question with a single word: 'yes' or 'no'.\n\n"
    "Question: {question}\n"
    "Answer (yes or no):"
)
baseline_prompt = ChatPromptTemplate.from_template(BASELINE_TEMPLATE)
baseline_chain = baseline_prompt | llm | StrOutputParser()

In [25]:
from tqdm.auto import tqdm

# Evaluate on a subset for speed; set N = len(questions) for full eval.
N = min(200, len(questions))
# Keep first N questions as evaluation subset
subset = questions.iloc[:N].reset_index(drop=False).rename(columns={"index": "orig_index"})
# Buffers to store predictions, retrieved doc ids, and raw model outputs
rag_preds, baseline_preds, retrieved_ids, raw_rag, raw_base = [], [], [], [], []

# Iterate through each question with a progress bar.
for _, row in tqdm(subset.iterrows(), total=len(subset)):
    q = row.question
    try:
        r = rag_chain.invoke(q)
        rag_text = r["answer"]
        # top-1 retrieved document id
        rid = r["context"][0].metadata.get("id") if r["context"] else None
    except Exception as e:
        # Fallback on failure so evaluation continues
        rag_text, rid = "", None
    try:
        base_text = baseline_chain.invoke({"question": q})
    except Exception:
        base_text = ""
    raw_rag.append(rag_text); raw_base.append(base_text)
    rag_preds.append(parse_yes_no(rag_text))
    baseline_preds.append(parse_yes_no(base_text))
    retrieved_ids.append(rid)

subset["rag_pred"] = rag_preds
subset["baseline_pred"] = baseline_preds
subset["retrieved_id"] = retrieved_ids
subset["rag_raw"] = raw_rag
subset["baseline_raw"] = raw_base

100%|██████████| 200/200 [03:38<00:00,  1.09s/it]


In [26]:
from sklearn.metrics import accuracy_score, f1_score

def score(df, pred_col):
    valid = df[df[pred_col].isin(["yes", "no"])]
    n_total, n_valid = len(df), len(valid)
    if n_valid == 0:
        return {"valid_rate": 0.0, "acc": None, "f1": None, "n_valid": 0, "n_total": n_total}
    y_true = valid.gold_label.tolist()
    y_pred = valid[pred_col].tolist()
    return {
        "valid_rate": n_valid / n_total,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label="yes"),
        "n_valid": n_valid,
        "n_total": n_total,
    }

print("RAG     :", score(subset, "rag_pred"))
print("Baseline:", score(subset, "baseline_pred"))

RAG     : {'valid_rate': 1.0, 'acc': 0.725, 'f1': 0.8, 'n_valid': 200, 'n_total': 200}
Baseline: {'valid_rate': 1.0, 'acc': 0.59, 'f1': 0.6870229007633588, 'n_valid': 200, 'n_total': 200}


## Alternative:

In [27]:
def classify_answer(text):
    t = text.strip().lower()
    if t.startswith("yes"):
        return "yes"
    if t.startswith("no"):
        return "no"
    return "invalid"

In [28]:
BASELINE_TEMPLATE = (
    "You are a medical question-answering assistant. "
    "Answer with a single word: yes or no.\n\n"
    "Question: {question}\n"
    "Answer (yes or no):"
)
baseline_prompt = ChatPromptTemplate.from_template(BASELINE_TEMPLATE)
baseline_chain = baseline_prompt | llm | StrOutputParser()

In [29]:
from tqdm.auto import tqdm

N = min(200, len(questions))
subset = questions.iloc[:N].reset_index(drop=True)

gold_labels = []
rag_preds = []
baseline_preds = []
retrieved_ids = []
rag_raws = []
baseline_raws = []
# Loop through each question
for i in tqdm(range(N)):
    q = subset.loc[i, "question"]
    gold = subset.loc[i, "gold_label"]

    rag_out = rag_chain.invoke(q)
    rag_text = rag_out["answer"]
    # Extract top-1 retrieved document id (if context exists)
    rid = rag_out["context"][0].metadata.get("id") if rag_out["context"] else None

    base_text = baseline_chain.invoke({"question": q})

    gold_labels.append(gold)
    rag_preds.append(classify_answer(rag_text))
    baseline_preds.append(classify_answer(base_text))
    retrieved_ids.append(rid)
    rag_raws.append(rag_text)
    baseline_raws.append(base_text)

subset["rag_pred"] = rag_preds
subset["baseline_pred"] = baseline_preds
subset["retrieved_id"] = retrieved_ids
subset["rag_raw"] = rag_raws
subset["baseline_raw"] = baseline_raws

100%|██████████| 200/200 [01:55<00:00,  1.74it/s]


In [30]:
from sklearn.metrics import accuracy_score, f1_score

def score(y_true, y_pred):
    # Count invalid answers as wrong (treat them as "no" so sklearn still works).
    n_invalid = sum(1 for p in y_pred if p == "invalid")
    y_pred_clean = ["no" if p == "invalid" else p for p in y_pred]
    acc = accuracy_score(y_true, y_pred_clean)
    f1 = f1_score(y_true, y_pred_clean, pos_label="yes")
    return {"accuracy": acc, "f1": f1, "invalid": n_invalid, "total": len(y_true)}

print("RAG     :", score(gold_labels, rag_preds))
print("Baseline:", score(gold_labels, baseline_preds))

RAG     : {'accuracy': 0.725, 'f1': 0.8, 'invalid': 0, 'total': 200}
Baseline: {'accuracy': 0.575, 'f1': 0.6743295019157088, 'invalid': 0, 'total': 200}


## My Reflection:
The results shows that RAG improve accuracy and f1 in comparison wit the baseline.

### Task 5.2. Detailed inspection: retrieval accuracy + qualitative check

In [31]:
# Retrieval accuracy: top-1 retrieved doc id vs. gold doc id
subset["retrieval_hit"] = subset.retrieved_id == subset.gold_document_id
print("Top-1 retrieval accuracy:", subset.retrieval_hit.mean())

Top-1 retrieval accuracy: 0.99


In [32]:
# Top-k retrieval accuracy for a few k
from collections import defaultdict

ks = [1, 3, 5, 10]
hits = defaultdict(int)
for _, row in subset.iterrows():
    docs = vector_store.similarity_search(row.question, k=max(ks))
    ids = [d.metadata.get("id") for d in docs]
    for k in ks:
        if row.gold_document_id in ids[:k]:
            hits[k] += 1
for k in ks:
    print(f"Recall@{k}: {hits[k] / len(subset):.3f}")

Recall@1: 0.990
Recall@3: 1.000
Recall@5: 1.000
Recall@10: 1.000


In [33]:
# Qualitative inspection of a few examples
for _, row in subset.head(5).iterrows():
    print("=" * 80)
    print("Q     :", row.question)
    print("Gold  :", row.gold_label, "  | RAG:", row.rag_pred, "  | Baseline:", row.baseline_pred)
    print("Retrieved id:", row.retrieved_id, " Gold id:", row.gold_document_id, " hit=", row.retrieval_hit)
    print("RAG raw     :", (row.rag_raw or "").strip()[:200])

Q     : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold  : yes   | RAG: yes   | Baseline: no
Retrieved id: 21645374  Gold id: 21645374  hit= True
RAG raw     : Yes
Q     : Landolt C and snellen e acuity: differences in strabismus amblyopia?
Gold  : no   | RAG: yes   | Baseline: yes
Retrieved id: 16418930  Gold id: 16418930  hit= True
RAG raw     : Yes
Q     : Syncope during bathing in infants, a pediatric form of water-induced urticaria?
Gold  : yes   | RAG: yes   | Baseline: no
Retrieved id: 9488747  Gold id: 9488747  hit= True
RAG raw     : Yes
Q     : Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?
Gold  : no   | RAG: yes   | Baseline: no
Retrieved id: 17208539  Gold id: 17208539  hit= True
RAG raw     : Yes
Q     : Can tailored interventions increase mammography use among HMO women?
Gold  : yes   | RAG: no   | Baseline: yes
Retrieved id: 10808977  Gold id: 10808977  hit= True


In [34]:
# Find all rows where either RAG or Baseline disagrees with the gold label.
mismatches = subset[(subset.rag_pred != subset.gold_label) | (subset.baseline_pred != subset.gold_label)]

print("Total mismatches:", len(mismatches), "out of", len(subset))
print("RAG wrong      :", (subset.rag_pred != subset.gold_label).sum())
print("Baseline wrong :", (subset.baseline_pred != subset.gold_label).sum())
print("Both wrong     :", ((subset.rag_pred != subset.gold_label) & (subset.baseline_pred != subset.gold_label)).sum())
print("Only RAG wrong :", ((subset.rag_pred != subset.gold_label) & (subset.baseline_pred == subset.gold_label)).sum())
print("Only Base wrong:", ((subset.rag_pred == subset.gold_label) & (subset.baseline_pred != subset.gold_label)).sum())

for _, row in mismatches.iterrows():
    print("=" * 80)
    print("Q     :", row.question)
    print("Gold  :", row.gold_label, "  | RAG:", row.rag_pred, "  | Baseline:", row.baseline_pred)
    print("Retrieved id:", row.retrieved_id, " Gold id:", row.gold_document_id, " hit=", row.retrieval_hit)
    print("RAG raw     :", (row.rag_raw or "").strip()[:200])
    print("Base raw    :", (row.baseline_raw or "").strip()[:200])

Total mismatches: 105 out of 200
RAG wrong      : 55
Baseline wrong : 85
Both wrong     : 35
Only RAG wrong : 20
Only Base wrong: 50
Q     : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold  : yes   | RAG: yes   | Baseline: no
Retrieved id: 21645374  Gold id: 21645374  hit= True
RAG raw     : Yes
Base raw    : No
Q     : Landolt C and snellen e acuity: differences in strabismus amblyopia?
Gold  : no   | RAG: yes   | Baseline: yes
Retrieved id: 16418930  Gold id: 16418930  hit= True
RAG raw     : Yes
Base raw    : Yes
Q     : Syncope during bathing in infants, a pediatric form of water-induced urticaria?
Gold  : yes   | RAG: yes   | Baseline: no
Retrieved id: 9488747  Gold id: 9488747  hit= True
RAG raw     : Yes
Base raw    : No
Q     : Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?
Gold  : no   | RAG: yes   | Baseline: no
Retrieved id: 17208539  Gold id: 17208539  hit= True
RAG r

To check whether increasing k above 1 can reduce the number of “Only RAG wrong” cases.

In [35]:
# Look only at "Only RAG wrong" cases and re-run them with more retrieved chunks.
only_rag_wrong = subset[(subset.rag_pred != subset.gold_label) & (subset.baseline_pred == subset.gold_label)]
print("Cases to re-test:", len(only_rag_wrong))

ks_to_try = [1, 3, 5, 10]

# 1) Does the gold document appear in top-k? (retrieval side)
for k in ks_to_try:
    hits = 0
    for _, row in only_rag_wrong.iterrows():
        docs = vector_store.similarity_search(row.question, k=k)
        ids = [d.metadata.get("id") for d in docs]
        if row.gold_document_id in ids:
            hits += 1
    print(f"Gold doc in top-{k}: {hits}/{len(only_rag_wrong)}")

Cases to re-test: 20
Gold doc in top-1: 19/20
Gold doc in top-3: 20/20
Gold doc in top-5: 20/20
Gold doc in top-10: 20/20


## My Reflection:
Seems retrieval is not the bottleneck. Lets test if with more K the performance of LLM improved for the cases that only RAG was wrong.

In [36]:
# 2) Does the LM answer correctly if we feed it top-k chunks concatenated?
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def build_rag_chain_with_k(k):
    retr = vector_store.as_retriever(search_kwargs={"k": k})
    ans = (
        {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
        | prompt
        | llm
        | StrOutputParser()
    )
    return RunnableParallel({"context": retr, "question": RunnablePassthrough()}).assign(answer=ans)

for k in ks_to_try:
    chain_k = build_rag_chain_with_k(k)
    fixed = 0
    for _, row in only_rag_wrong.iterrows():
        out = chain_k.invoke(row.question)
        pred = classify_answer(out["answer"])
        if pred == row.gold_label:
            fixed += 1
    print(f"k={k}: {fixed}/{len(only_rag_wrong)} fixed")

k=1: 0/20 fixed
k=3: 19/20 fixed
k=5: 18/20 fixed
k=10: 15/20 fixed


## My Reflection:
Based on the results k=3 is the most suitable number.

**Discussion.**
- *Did retrieval help?* Compare the RAG and baseline rows above: with a small LM the no-context baseline often defaults to one label or hallucinates, while RAG conditions on a relevant abstract and typically improves accuracy/F1 on the valid subset.
- *Retrieval quality.* Recall@1 is typically modest because PubMedQA questions are paraphrases of the abstract, but Recall@5/@10 is much higher. Increasing `k` in the retriever and concatenating multiple chunks usually helps the LM, at the cost of longer prompts.
- *Failure modes to look for in the qualitative dump:* gold doc not retrieved, gold doc retrieved but LM ignores it, LM produces text that does not contain `yes`/`no` (caught by `parse_yes_no`).

## My Reflection:

The results show that RAG improves accuracy and F1 in comparison with the baseline.

Also, it seems retrieval is not the main bottleneck. To verify this, I tested higher values of K on the cases where only RAG was wrong, to see whether LLM performance improves when more chunks are provided.

Based on these results, k = 3 is the most suitable number.

Main failure cases I saw:
- gold document is not retrieved,
- gold document is retrieved but the LLM still does not use it well,
- output is not clean yes/no (handled by parse_yes_no).

So overall, retrieval improves performance, but ranking quality and answer formatting are still important bottlenecks.